# Обучение DnCNN-Lite (Shumodav)

Запускай ячейки по одной сверху вниз. Инструкция: `docs/TRAINING_COLAB.md`. Включи GPU: Среда выполнения → Сменить среду выполнения → T4 GPU.

## 1. Проверка видеокарты

In [ ]:
!nvidia-smi
import torch
print("CUDA доступна:", torch.cuda.is_available())

## 2. Google Drive для чекпойнтов

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CKPT = '/content/drive/MyDrive/shumodav_ckpt'
!mkdir -p {CKPT}

## 3. Код проекта

In [ ]:
!rm -rf /content/shumodav
!git clone -b v2 https://github.com/mkass420/shumodav.git /content/shumodav
%cd /content/shumodav
!pip install -q -e . --no-deps --ignore-requires-python
!pip install -q scikit-image tqdm

## 4. Данные

In [ ]:
!python scripts/get_data.py
!ls data/train400 | wc -l   # должно быть 400
!ls data/bsd68 | wc -l      # должно быть 68

## 5. Пробный запуск (1–2 минуты)

In [ ]:
!python -m shumodav.nn.train --smoke --device cuda --out /content/smoke.pth

## 6. Обучение (20–40 минут)

Если сессия оборвалась: запусти ячейки 1–4 и эту ещё раз, `--resume` продолжит с чекпойнта на Drive.

In [ ]:
!python -m shumodav.nn.train --data data/train400 --val data/bsd68 \
    --iters 15000 --batch 128 --lr 1e-3 --device cuda \
    --ckpt-dir {CKPT} --out {CKPT}/custom_cnn.pth --resume

## 7. График обучения

In [ ]:
from IPython.display import Image, display
import pandas as pd
display(Image(f'{CKPT}/train_curve.png'))
pd.read_csv(f'{CKPT}/train_log.csv').tail(5)

## 8. Скачать результаты

In [ ]:
from google.colab import files
for name in ['custom_cnn.pth', 'train_log.csv', 'train_curve.png']:
    files.download(f'{CKPT}/{name}')